TSP & SPP COUPLED CODE


In [9]:
import sys
from pathlib import Path
import numpy as np
import rasterio as rio
from rasterio.plot import show
import matplotlib.pyplot as plt
from time import perf_counter
import pandas as pd
import elevation 
import random
import copy
import datetime
import itertools
import math
import cbor2
from astropy.time import Time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'BORAS_v1.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import BORAS_v1

In [10]:
class path_info:
        def __init__(self,name,eu_fit,r_fit,bpath,t_trav,s_time):
            self.name = name
            self.eu_fit = eu_fit
            self.r_fit = r_fit
            self.bpath = bpath
            self.t_trav = t_trav
            self.s_time = s_time
            
def crossover3(parent1,parent2,crossover_rate=0.9):
    if random.random()< crossover_rate:
            crossover_point= random.randint(0,(len(parent1)-1))
            child = parent1[:crossover_point]+(parent2[crossover_point:])
            return child
    else:
        return parent1
    
def mutation2(path, mutation_rate=0.1):
    if random.random()< mutation_rate:
            mutation_point1= random.randint(0,(len(path)))
            mutation_point2= random.randint(0,(len(path)))
            place_holder = path[mutation_point1]
            path[mutation_point1] = path[mutation_point2]
            path[mutation_point2] = place_holder
    return path

def canonical_form(array):
    n = len(array)
    
    # Create all rotations of the array
    rotations = [array[i:] + array[:i] for i in range(n)]
    
    # Create all rotations of the reversed array
    reversed_array = array[::-1]
    reverse_rotations = [reversed_array[i:] + reversed_array[:i] for i in range(n)]
    
    # Combine and find the lexicographically smallest rotation
    all_rotations = rotations + reverse_rotations
    smallest_rotation = min(all_rotations)
    
    return smallest_rotation

def uniform_based_order(individual_1, individual_2, shuffle_size=5):
    """
    uniform based order crossover
    
    :param individual_1: 
    :param individual_2: 
    :param shuffle_size: 
    :return: two children
    """

    child_1 = copy.deepcopy(individual_1)
    child_2 = copy.deepcopy(individual_2)

    size = min(len(child_1), len(child_2))
    if size < shuffle_size:
        shuffle_size = size - 1

    index = [*range(size)]
    bit_mask = set(random.sample(index, shuffle_size))

    shuffle_1 = [individual_1[idx] for idx in bit_mask]
    shuffle_2 = [individual_2[idx] for idx in bit_mask]

    inv_bit_mask = set(index) - bit_mask

    parent1 = [individual_1[i] for i in inv_bit_mask]
    parent2 = [individual_2[i] for i in inv_bit_mask]

    order_shuffle_1 = set(shuffle_1)
    order_shuffle_2 = set(shuffle_2)

    can_shuffle = order_shuffle_1 & order_shuffle_2
    remain_shuffle = order_shuffle_1 ^ order_shuffle_2

    # create order from data
    order_shuffle_1 = [x for x in shuffle_1 if x in can_shuffle]
    order_shuffle_2 = [x for x in shuffle_2 if x in can_shuffle]

    parent1 = [x for x in parent1 if x in remain_shuffle]
    parent2 = [x for x in parent2 if x in remain_shuffle]

    order_shuffle_1 += parent1
    order_shuffle_2 += parent2

    i = 0
    for idx in bit_mask:
        child_1[idx] = order_shuffle_2[i]
        child_2[idx] = order_shuffle_1[i]
        i += 1

    return child_1, child_2

In [ ]:

# Use the function to read the input parameters
input_file = PROJECT_ROOT / 'configs' / 'input_config2.txt'
params = BORAS_v1.parse_input_file(input_file)

# Assign the parameters to variables, with default values if needed
mean_speed = params.get('mean_speed', 0.0175)  # Default to 0.0175 if not provided
start_epoch = params.get('start_epoch', [])
start_time_range_start = params.get('start_time_range_start', [])
start_time_range_end = params.get('start_time_range_end', [])
start_time_range_resolution = params.get('start_time_range_resolution', 0.1)

# Only generate the time range if all values are provided
if start_time_range_start is not None and start_time_range_end is not None and start_time_range_resolution is not None:
    start_time_range = BORAS_v1.generate_time_range(start_time_range_start, start_time_range_end, start_time_range_resolution)
else:
    start_time_range = None

time_dependent = params.get('time_dependent', False)
mode = params.get('mode', 'Single')
tsp_mode = params.get('tsp_mode', 'OL')
algorithm = params.get('algorithm', 'Dijkistra')
pop_size = params.get('pop_size', 16)
max_generations = params.get('max_generations', 100)
filename = params.get('filename', [])
illumination_filename = params.get('illumination_filename', [])
waypoints = params.get('waypoints', [])

if filename and not Path(filename).is_absolute():
    filename = str(PROJECT_ROOT / filename)
if illumination_filename and not Path(illumination_filename).is_absolute():
    illumination_filename = str(PROJECT_ROOT / illumination_filename)

# Open the DEM and illumination files if filenames are provided
if filename:
    dem = rio.open(filename)
else:
    dem = None

if illumination_filename:
    with open(illumination_filename, "rb") as file: 
        roots = cbor2.load(file)
else:
    roots = None

# Print the waypoints to verify they are read correctly
print(waypoints)

In [ ]:
if mode == 'Tour' and time_dependent == False:
    order = list(np.arange((len(waypoints))))    
    permutations = list(itertools.permutations(order,r=2))
    info_paths = []
    total_t_time = 0
    max_min = False
    for i in range(len(permutations)):
        start = waypoints[permutations[i][0]]
        end = waypoints[permutations[i][1]]
        distance=np.sqrt((start[0]-end[0])**2 + (start[1]-end[1])**2)
        t_time = distance/mean_speed
        info_paths.append(path_info(permutations[i],t_time,0,[],0,0))
elif  mode == 'Tour' and time_dependent == True:
    order = list(np.arange((len(waypoints))))  
    permutations = list(itertools.permutations(order,r=len(waypoints)))
    print(permutations)
    info_paths = []
    total_t_time = 0
    for i in range(len(permutations)):
        for j in range(len(waypoints)-1):
            start = waypoints[permutations[i][j]]
            end = waypoints[permutations[i][j+1]]
            distance=np.sqrt((start[0]-end[0])**2 + (start[1]-end[1])**2)
            t_time = distance/mean_speed
            total_t_time =total_t_time + t_time
        info_paths.append(path_info(permutations[i],total_t_time,0,[],[],[]))
elif mode == 'Tour' and algorithm == 'Dijkistra':
    order = list(np.arange((len(waypoints))))    
    permutations = list(itertools.permutations(order,r=2))
    info_paths = []
    total_t_time = 0
    max_min = False
    for i in range(len(permutations)):
        start = waypoints[permutations[i][0]]
        end = waypoints[permutations[i][1]]
        distance=np.sqrt((start[0]-end[0])**2 + (start[1]-end[1])**2)
        t_time = distance/mean_speed
        info_paths.append(path_info(permutations[i],t_time,0,[],0))


In [2]:
#TSP GA CODE
def tsp(waypoints,pop_size,max_generations,tsp_mode):
    generations = 0 #Initialize Generations
    pop = [] 
    fitness = []
    same = 0
    best_fitness = 0
    #GENERATE INITIAL POPULATION
    for indv in range(pop_size):
        random.shuffle(order)
        populant = copy.deepcopy(order)
        populant = canonical_form(populant)
        pop.append(populant)
        total_fit = 0
        for i in range(len(waypoints)):
            idx_perm = permutations.index((order[(i-1)],order[i]))
            if info_paths[idx_perm].r_fit == 0:
                fit = info_paths[idx_perm].eu_fit
            else:
                fit = info_paths[idx_perm].r_fit
            total_fit = total_fit + fit
        total_fit = 1-(1/total_fit)
        fitness.append(total_fit)

    while not BORAS_v1.termination_condition(generations,max_generations)  or same>= 33:
        generations += 1
        next_gen = []
        old_best_fitness = best_fitness
        #SELECT POPULATION FOR NEXT GEN
        selected_pop_size = int(pop_size/8)
        select_fitness = copy.deepcopy(fitness)
        print(select_fitness)
        pop_w_elite = copy.deepcopy(pop)
        elite_pop = BORAS_v1.select_pop_max2(pop,select_fitness, pop_size,max_min, num_to_select=selected_pop_size)
        select_fitness = copy.deepcopy(fitness)
        selected_pop = BORAS_v1.select_pop_rand(pop_w_elite,select_fitness, pop_size, selected_pop_size*6)
        selected_pop = selected_pop +  elite_pop
        
        pop_indexes = list(np.arange(0,pop_size,1,dtype=int))

        #ELITE
        for i in range(selected_pop_size):
            next_gen.append(elite_pop[i])

        #CROSSOVER & MUTATION
        pop_indexes = list(np.arange(0,( selected_pop_size*6),1,dtype=int))
        random.shuffle(selected_pop)
        for _ in range(0, ( selected_pop_size*6),2):
                    i,j = random.sample(pop_indexes,k=2)
                    pop_indexes.remove(i)
                    pop_indexes.remove(j)
                    parent1 =[]
                    parent2=[]
                    child1=[]
                    child2=[]
                    parent1 , parent2 = selected_pop[i],selected_pop[j]
                    child1,child2 = uniform_based_order(parent1,parent2)#crossover3(parent1,parent2), crossover3(parent2,parent1)
                    #MUTATION
                    child1,child2 = BORAS_v1.mutation2(child1),BORAS_v1.mutation2(child2)
                    child1 = canonical_form(child1)
                    child2 = canonical_form(child2)
                    next_gen.append(child1)
                    next_gen.append(child2)


        for indv in range(selected_pop_size):
            random.shuffle(order)
            populant = copy.deepcopy(order)
            populant = canonical_form(populant)
            next_gen.append(populant)

        pop[:] = next_gen
        fitness = []
        print(pop)
        #EVALUATE POPULATION
        for path in pop:
            total_fit = 0
            if tsp_mode == 'CL':
                for i in range(len(path)):
                    idx_perm = permutations.index((path[(i-1)],path[i]))
                    if info_paths[idx_perm].r_fit == 0:
                        fit = info_paths[idx_perm].eu_fit
                    else:
                        fit = info_paths[idx_perm].r_fit
                    total_fit = total_fit + fit
            elif tsp_mode == 'OL':
                for i in range(len(path)-1):
                    idx_perm = permutations.index((path[i],path[(i+1)]))
                    if info_paths[idx_perm].r_fit == 0:
                        fit = info_paths[idx_perm].eu_fit
                    else:
                        fit = info_paths[idx_perm].r_fit
                    total_fit = total_fit + fit
            total_fit = float(1-(1/total_fit))
            fitness.append(total_fit)       
        best_fitness = min(fitness)
        best_path_idx = fitness.index(best_fitness)
        print(f'Generation {generations}: Best fitness = {best_fitness}.')
        best_path = pop[best_path_idx]
    if best_fitness == old_best_fitness:
        same += 1
    else:
        same = 0
    return best_path, best_fitness

def td_tsp(waypoints,pop_size,max_generations,tsp_mode,info_paths,start_epoch,start_time_range_resolution):
    slope_dem = dem.read(1).astype('float64')
    generations = 0 #Initialize Generations
    pop = [] 
    fitness = []
    same = 0
    best_fitness = 0
    #GENERATE INITIAL POPULATION
    for indv in range(pop_size):
        random.shuffle(order)
        populant = copy.deepcopy(order)
        while populant  in pop:
            random.shuffle(order)
            populant = copy.deepcopy(order)
        pop.append(populant)
        populant = tuple(populant)
        total_fit = 0
        start_time_ranges = copy.deepcopy(start_time_range)
        idx_perm = permutations.index(populant)
        if tsp_mode == "OL":
            for i in range(len(populant)-1):
                dem_c = []
                start = waypoints[populant[i]]
                end = waypoints[populant[i+1]]
                if algorithm == "GA":
                    bfit_ga_path,fitnesses_gen,traverse_time,best_ga_path = BORAS_v1.ga(start,end,dem,start_epoch, roots,time_dependent)
                    if not np.isnan(traverse_time):
                        time = Time(start_epoch, format='iso', scale='tdb')
                        start_time = time.cxcsec
                        info_paths[idx_perm].s_time.append(start_time)  # Append before modifying `start_time`
                        start_time += traverse_time  # Update start_time after appending the original value    time = Time(start_time, format='cxcsec', scale='tdb')
                        start_time = Time(start_time, format='cxcsec', scale='tdb')
                        start_epochs = start_time.iso
                        fit = traverse_time
                    else:
                        fit = np.inf
                elif algorithm == "TDD":
                    start_time, best_ga_path, traverse_time,path_times = BORAS_v1.TDD_expanded(start,end,slope_dem,roots,start_time_ranges)
                    if not np.isnan(traverse_time):
                        info_paths[idx_perm].s_time.append(start_time)
                        start_time = start_time + (traverse_time)
                        fit = (traverse_time*24*60*60)
                        start_time_ranges = BORAS_v1.generate_time_range(start_time,start_time_ranges[-1],start_time_range_resolution)
                    else:
                        fit = np.inf
                        print(start_time_ranges)
                info_paths[idx_perm].bpath.append(best_ga_path)
                info_paths[idx_perm].t_trav.append(traverse_time)
                total_fit = total_fit + fit
        elif tsp_mode == "CL":
            for i in range(len(populant)):
                dem_c = []
                start = waypoints[populant[i-1]]
                end = waypoints[populant[i]]
                if algorithm == "GA":
                    bfit_ga_path,fitnesses_gen,traverse_time,best_ga_path = BORAS_v1.ga(start,end,dem,start_epoch, roots,time_dependent)
                    if not np.isnan(traverse_time):
                        time = Time(start_epoch, format='iso', scale='tdb')
                        start_time = time.cxcsec
                        info_paths[idx_perm].s_time.append(start_time)  # Append before modifying `start_time`
                        start_time += traverse_time  # Update start_time after appending the original value    time = Time(start_time, format='cxcsec', scale='tdb')
                        start_time = Time(start_time, format='cxcsec', scale='tdb')
                        start_epochs = start_time.iso
                        fit = traverse_time
                    else:
                        fit = np.inf
                elif algorithm == "TDD":
                    start_time, best_ga_path, traverse_time,path_times = BORAS_v1.TDD_expanded(start,end,slope_dem,roots,start_time_ranges)
                    print(idx_perm)
                    if not np.isnan(traverse_time):
                        info_paths[idx_perm].s_time.append(start_time)
                        start_time = start_time + (traverse_time)
                        fit = (traverse_time*24*60*60)
                        start_time_ranges = BORAS_v1.generate_time_range(start_time,start_time_ranges[-1],start_time_range_resolution)
                    else:
                        fit = np.inf
                info_paths[idx_perm].bpath.append(best_ga_path)
                info_paths[idx_perm].t_trav.append(traverse_time)
                total_fit = total_fit + fit
        total_fit = float(1-(1/total_fit))
        info_paths[idx_perm].r_fit = total_fit
        fitness.append(total_fit)     
    
    while not BORAS_v1.termination_condition(generations,max_generations)  or same>= 33:
        generations += 1
        next_gen = []
        max_min = False
        old_best_fitness = best_fitness
        #SELECT POPULATION FOR NEXT GEN
        selected_pop_size = int(pop_size/8)
        select_fitness = copy.deepcopy(fitness)
        print(select_fitness)
        pop_w_elite = copy.deepcopy(pop)
        elite_pop = BORAS_v1.select_pop_max2(pop,select_fitness, pop_size,max_min, num_to_select=selected_pop_size)
        select_fitness = copy.deepcopy(fitness)
        selected_pop = BORAS_v1.select_pop_rand(pop_w_elite,select_fitness, pop_size, selected_pop_size*6)
        selected_pop = selected_pop +  elite_pop
        
        pop_indexes = list(np.arange(0,pop_size,1,dtype=int))

        #ELITE
        for i in range(selected_pop_size):
            next_gen.append(elite_pop[i])

        #CROSSOVER & MUTATION
        pop_indexes = list(np.arange(0,( selected_pop_size*6),1,dtype=int))
        random.shuffle(selected_pop)
        for _ in range(0, ( selected_pop_size*6),2):
                    i,j = random.sample(pop_indexes,k=2)
                    pop_indexes.remove(i)
                    pop_indexes.remove(j)
                    parent1 =[]
                    parent2=[]
                    child1=[]
                    child2=[]
                    parent1 , parent2 = selected_pop[i],selected_pop[j]
                    child1,child2 = uniform_based_order(parent1,parent2)#crossover3(parent1,parent2), crossover3(parent2,parent1)
                    #MUTATION
                    child1,child2 = BORAS_v1.mutation2(child1),BORAS_v1.mutation2(child2)
                    if tsp_mode == "CL":
                        child1 = canonical_form(child1)
                        child2 = canonical_form(child2)
                    next_gen.append(child1)
                    next_gen.append(child2)


        for indv in range(selected_pop_size):
            random.shuffle(order)
            populant = copy.deepcopy(order)
            if tsp_mode == "CL":
                populant = canonical_form(populant)
            next_gen.append(populant)

        pop[:] = next_gen
        fitness = []
        print(pop)
        #EVALUATE POPULATION
        for path in pop:
            total_fit = 0
            start_time_ranges = copy.deepcopy(start_time_range)
            start_epochs = copy.deepcopy(start_epoch)
            if tsp_mode == 'CL':
                path = tuple(path)
                idx_perm = permutations.index(path)
                if info_paths[idx_perm].r_fit != 0 :
                    total_fit = float(info_paths[idx_perm].r_fit) 
                else:
                    for i in range(len(path)):
                        start_time,best_ga_path,traverse_time = None, None, None
                        idx_perm = permutations.index(path)
                        dem_c = []
                        start = waypoints[path[(i-1)]]
                        end = waypoints[path[i]]
                        if algorithm == "GA":
                            bfit_ga_path,fitnesses_gen,traverse_time,best_ga_path = BORAS_v1.ga(start,end,dem,start_epochs, roots,time_dependent)
                            if not np.isnan(traverse_time):
                                time = Time(start_epoch, format='iso', scale='tdb')
                                start_time = time.cxcsec
                                info_paths[idx_perm].s_time.append(start_time)  # Append before modifying `start_time`
                                start_time += traverse_time  # Update start_time after appending the original value    time = Time(start_time, format='cxcsec', scale='tdb')
                                start_time = Time(start_time, format='cxcsec', scale='tdb')
                                start_epochs = start_time.iso
                                fit = traverse_time
                            else:
                                total_fit = np.inf
                                print("Traverse time is NaN. Skipping update.")
                            
                        elif algorithm == "TDD":
                            start_time, best_ga_path, traverse_time,path_times = BORAS_v1.TDD_expanded(start,end,slope_dem,roots,start_time_ranges)
                            if start_time == None:
                                fit = np.inf
                                start_time = 0
                            info_paths[idx_perm].s_time.append(start_time)
                            start_time = start_time + traverse_time
                            start_time_ranges= BORAS_v1.generate_time_range(start_time,start_time_ranges[-1],start_time_range_resolution)
                            fit = (traverse_time*24*60*60)
                        info_paths[idx_perm].bpath.append(best_ga_path)
                        info_paths[idx_perm].t_trav.append(traverse_time)
                        total_fit = total_fit + fit
                    total_fit = float(1-(1/total_fit))
                    info_paths[idx_perm].r_fit = total_fit
                fitness.append(total_fit)

            elif tsp_mode == 'OL':
                path = tuple(path)
                idx_perm = permutations.index(path)
                if info_paths[idx_perm].r_fit != 0 :
                    total_fit = float(info_paths[idx_perm].r_fit) 
                else:
                    for i in range(len(path)-1):
                        start_time,best_ga_path,traverse_time = None, None, None
                        start = waypoints[path[i]]
                        end = waypoints[path[(i+1)]]
                        if algorithm == "GA":
                            bfit_ga_path,fitnesses_gen,traverse_time,best_ga_path = BORAS_v1.ga(start,end,dem,start_epochs, roots, time_dependent)
                            if not np.isnan(traverse_time):
                                time = Time(start_epoch, format='iso', scale='tdb')
                                start_time = time.cxcsec
                                info_paths[idx_perm].s_time.append(start_time)  # Append before modifying `start_time`
                                start_time += traverse_time  # Update start_time after appending the original value    time = Time(start_time, format='cxcsec', scale='tdb')
                                start_time = Time(start_time, format='cxcsec', scale='tdb')
                                start_epochs = start_time.iso
                                fit = traverse_time
                            else:
                                fit = np.inf
                        elif algorithm == "TDD":
                            start_time, best_ga_path, traverse_time,path_times = BORAS_v1.TDD_expanded(start,end,slope_dem,roots,start_time_ranges)
                            if not np.isnan(start_time):
                                info_paths[idx_perm].s_time.append(start_time)
                                start_time = start_time + (traverse_time)
                                fit = (traverse_time*24*60*60)
                                start_time_ranges = BORAS_v1.generate_time_range(start_time,start_time_ranges[-1],start_time_range_resolution)
                                print(start_time_ranges)
                            else:
                                fit = np.inf
                        info_paths[idx_perm].bpath.append(best_ga_path)
                        info_paths[idx_perm].t_trav.append(traverse_time) 
                        total_fit = total_fit + fit
                    total_fit = float(1-(1/total_fit))
                    info_paths[idx_perm].r_fit = total_fit
                fitness.append(total_fit)      
        best_fitness = min(fitness)
        best_path_idx = fitness.index(best_fitness)
        print(f'Generation {generations}: Best fitness = {best_fitness}.')
        best_path = tuple(pop[best_path_idx])
        idx_best = permutations.index(best_path)
        traverse_times = info_paths[idx_best].t_trav
        opt_start_times = info_paths[idx_best].s_time
    if best_fitness == old_best_fitness:
        same += 1
    else:
        same = 0
    return traverse_times,opt_start_times,best_path, best_fitness,info_paths



In [ ]:
if mode == 'Tour' and time_dependent == False:
    best_path, best_fitness = tsp(waypoints,pop_size,max_generations,tsp_mode)
    print('Best path found:',best_path)
    old_best_fitness = 10
    old_best_path =[]
    gen_eq = 0
    #best_path = [0,1,2,3,4]
    while (old_best_fitness != best_fitness) or (best_path!=old_best_path):
        old_best_fitness = best_fitness
        old_best_path = best_path
        if tsp_mode == 'CL':
            for i in range(len(best_path)):
                idx_perm = permutations.index((best_path[(i-1)],best_path[i]))
                idx_perm_r = permutations.index((best_path[i],best_path[(i-1)]))
                if info_paths[idx_perm].r_fit == 0:
                    dem_c = []
                    start = waypoints[best_path[(i-1)]]
                    end = waypoints[best_path[i]]
                    new_filename = "crop-file"
                    BORAS_v1.crop_grid(start,end,filename,new_filename)
                    dem_c = rio.open(new_filename)
                    dem_array = dem_c.read(1).astype('float64')
                    new_start, new_end = BORAS_v1.new_start_end(start,end,dem_c)
                    if algorithm == "GA":
                        bfit_ga_path,fitnesses_gen,traverse_time,best_ga_path = BORAS_v1.ga(new_start,new_end,dem,start_epoch, roots,time_dependent)
                    elif algorithm == "Dijkistra":
                        traverse_time,best_ga_path = BORAS_v1.djikistras(new_start,new_end,dem_array)
                    info_paths[idx_perm].r_fit = traverse_time
                    info_paths[idx_perm_r].r_fit = traverse_time
                    best_ga_path_trans = BORAS_v1.translate_back_to_original(start, end, best_ga_path)
                    info_paths[idx_perm].bpath = best_ga_path_trans
                    info_paths[idx_perm_r].bpath = best_ga_path_trans
                    info_paths[idx_perm].t_trav = traverse_time
                    info_paths[idx_perm_r].t_trav = traverse_time
                    dem_c.close()
        elif tsp_mode == 'OL':
            for i in range(len(best_path)-1):
                idx_perm = permutations.index((best_path[i],best_path[(i+1)]))
                idx_perm_r = permutations.index((best_path[(i+1)],best_path[i]))
                if info_paths[idx_perm].r_fit == 0:
                    dem_c = []
                    start = waypoints[best_path[i]]
                    end = waypoints[best_path[(i+1)]]
                    new_filename = "crop-file"
                    BORAS_v1.crop_grid(start,end,filename,new_filename)
                    dem_c = rio.open(new_filename)
                    dem_array = dem_c.read(1).astype('float64')
                    new_start, new_end = BORAS_v1.new_start_end(start,end,dem_c)
                    if algorithm == "GA":
                        bfit_ga_path,fitnesses_gen,traverse_time,best_ga_path = BORAS_v1.ga(new_start,new_end,dem,start_epoch,roots,time_dependent)
                    elif algorithm == "Dijkistra":
                        traverse_time,best_ga_path = BORAS_v1.djikistras(new_start,new_end,dem_array)
                    info_paths[idx_perm].r_fit = traverse_time
                    info_paths[idx_perm_r].r_fit = traverse_time
                    best_ga_path_trans = BORAS_v1.translate_back_to_original(start, end,best_ga_path)
                    info_paths[idx_perm].bpath = best_ga_path_trans
                    info_paths[idx_perm_r].bpath = best_ga_path_trans
                    info_paths[idx_perm].t_trav = traverse_time
                    info_paths[idx_perm_r].t_trav = traverse_time
                    dem_c.close()
        best_path, best_fitness = tsp(waypoints,pop_size,max_generations,tsp_mode)
    print('Best path found:',best_path)
elif mode == 'Tour' and time_dependent == True:
    if algorithm == 'GA':
        traverse_time,opt_start_time,best_path, best_fitness,info_paths = td_tsp(waypoints,pop_size,max_generations,tsp_mode,info_paths,start_epoch,None)
    else:
        traverse_time,opt_start_time,best_path, best_fitness,info_paths = td_tsp(waypoints,pop_size,max_generations,tsp_mode,info_paths,start_time_range,start_time_range_resolution)
    print('Best path found:',best_path)
else:
    if algorithm == "GA":
        bfit_ga_path,fitnesses_gen,traverse_time,best_ga_path = BORAS_v1.ga(waypoints[0],waypoints[1],dem,start_epoch,roots,time_dependent)
        print('Best path found:',best_ga_path)
    elif algorithm == "Dijkistra":
        dem_array = dem.read(1).astype('float64')
        traverse_time,best_ga_path = BORAS_v1.djikistras(waypoints[0],waypoints[1],dem_array)
        print('Best path found:',best_ga_path)
    elif algorithm == "TDD":
        slope_dem = dem.read(1).astype('float64')
        opt_start_time, best_ga_path, traverse_time,path_times = BORAS_v1.TDD_expanded(waypoints[0],waypoints[1],slope_dem,roots,start_time_range)
        #opt_start_time, traverse_time, best_ga_path = BORAS_v1.TDD_non_FIFO(waypoints[0],waypoints[1], slope_dem, roots, start_time_range)
        print('Best path found:',best_ga_path)
        print(path_times)

In [ ]:
%matplotlib inline
import matplotlib.cm as cm
try:
    import mpld3
    mpld3.disable_notebook()
except ImportError:
    mpld3 = None
#best_path= [0, 1, 2, 3, 4]
dem = rio.open(filename)
slope_dem =dem.read(1)
if mode == 'Tour' and time_dependent == False:
    color = iter(cm.rainbow(np.linspace(0, 1, len(best_path))))
    fig, ax = plt.subplots(1, figsize=(5, 5))
    ax.imshow(slope_dem, origin='upper', cmap='Greys_r', interpolation='none')
    total_trav_time = 0
    if tsp_mode == "OL":
        for i in range(len(best_path)-1):
            trav_time = 0
            idx_perm = permutations.index((best_path[i],best_path[i+1]))
            best_ga_path = info_paths[idx_perm].bpath
            trav_time = info_paths[idx_perm].t_trav
            if info_paths[idx_perm].t_trav == 0:
                trav_time = info_paths[idx_perm].eu_fit
            total_trav_time =  total_trav_time + trav_time
                    # Extract x, y coordinates for path plotting
            path_x = [point[1] for point in best_ga_path]  # Col indices
            path_y = [point[0] for point in best_ga_path]  # Row indices
                # Plot path
            c = next(color)
            plt.plot(path_x, path_y, marker='o', color=c, markersize=1, linewidth=0.5)
            plt.title('Shortest Route Through the Waypoints')
    else:
        for i in range(len(best_path)):
            trav_time = 0
            idx_perm = permutations.index((best_path[i-1],best_path[i]))
            best_ga_path = info_paths[idx_perm].bpath
            trav_time = info_paths[idx_perm].t_trav
            if info_paths[idx_perm].t_trav == 0:
                trav_time = info_paths[idx_perm].eu_fit
            total_trav_time =  total_trav_time + trav_time
                    # Extract x, y coordinates for path plotting
            path_x = [point[1] for point in best_ga_path]  # Col indices
            path_y = [point[0] for point in best_ga_path]  # Row indices
                # Plot path
            c = next(color)
            plt.plot(path_x, path_y, marker='o', color=c, markersize=1, linewidth=0.5)
            plt.title('Shortest Route Through the Waypoints')
elif mode == "Tour" and time_dependent == True:
    color = iter(cm.rainbow(np.linspace(0, 1, len(best_path))))
    fig, ax = plt.subplots(1, figsize=(5, 5))
    ax.imshow(slope_dem, origin='upper', cmap='Greys_r', interpolation='none')
    total_trav_time = 0
    if tsp_mode == "OL":
        for i in range(len(best_path)-1):
            trav_time = 0
            idx_perm = permutations.index(best_path)
            best_ga_path = info_paths[idx_perm].bpath[i]
            trav_time = info_paths[idx_perm].t_trav[i]
            if info_paths[idx_perm].t_trav[i] == 0:
                trav_time = info_paths[idx_perm].eu_fit[i]
            total_trav_time =  total_trav_time + trav_time
                    # Extract x, y coordinates for path plotting
            path_x = [point[1] for point in best_ga_path]  # Col indices
            path_y = [point[0] for point in best_ga_path]  # Row indices
                # Plot path
            c = next(color)
            plt.plot(path_x, path_y, marker='o', color=c, markersize=1, linewidth=0.5)
            plt.title('Shortest Route Through the Waypoints')
    else:
        for i in range(len(best_path)):
            trav_time = 0
            idx_perm = permutations.index(best_path)
            best_ga_path = info_paths[idx_perm].bpath[i]
            trav_time = info_paths[idx_perm].t_trav[i]
            if info_paths[idx_perm].t_trav[i] == 0:
                trav_time = info_paths[idx_perm].eu_fit[i]
            total_trav_time =  total_trav_time + trav_time
                    # Extract x, y coordinates for path plotting
            path_x = [point[1] for point in best_ga_path]  # Col indices
            path_y = [point[0] for point in best_ga_path]  # Row indices
                # Plot path
            c = next(color)
            plt.plot(path_x, path_y, marker='o', color=c, markersize=1, linewidth=0.5)
            plt.title('Shortest Route Through the Waypoints')
else:
    fig, ax = plt.subplots(1, figsize=(5, 5))
    ax.imshow(slope_dem, origin='upper', cmap='Greys_r', interpolation='none')
    # Extract x, y coordinates for path plotting
    path_x = [point[1] for point in best_ga_path]  # Col indices
    path_y = [point[0] for point in best_ga_path]  # Row indices
    # Plot path
    plt.plot(path_x, path_y, marker='o', markersize=0.1, linewidth=2)
    plt.title('Shortest Route')
    total_trav_time = traverse_time
plt.xlabel('Column Index')
plt.ylabel('Row Index')
plt.grid(False)
plt.show()
print(total_trav_time)

In [ ]:
# Optional manual-debug cell retained for ad-hoc exploration.
print('Optional debug cell: run only when exploring a matching scenario manually.')

In [ ]:
# Optional manual-debug cell retained for ad-hoc exploration.
print('Optional debug cell: run only when exploring a matching scenario manually.')

In [ ]:
# Optional manual-debug cell retained for ad-hoc exploration.
print('Optional debug cell: run only when exploring a matching scenario manually.')

In [ ]:
# Optional manual-debug cell retained for ad-hoc exploration.
print('Optional debug cell: run only when exploring a matching scenario manually.')